# Normal maps

Experiments with bumping and normal mapping


In [ ]:
from typing import NamedTuple
import math
import numpy as np
from numpy.typing import NDArray, ArrayLike
import k3d
from k3d.colormaps.matplotlib_color_maps import Binary_r
from k3d import helpers as k3helpers

Vec = np.ndarray[tuple[int], np.dtype[np.floating]]  # 2D or 3D
vec = lambda *args: np.array(args, dtype=float)
asarr = np.asarray
nparr = lambda *args: np.array(args)
npgen = lambda iter: np.fromiter(iter, dtype=float)
empty = lambda: np.array(())
powers = lambda t: np.stack([np.ones(np.shape(t)), t, np.square(t)], axis=-1)  # tshape×3
dot = lambda v1, v2: float(v1 @ v2)
dots = lambda a1, a2: (a1 * a2).sum(axis=-1)
dot2 = lambda v: (v * v).sum(axis=-1)
normalize = lambda vecarr: vecarr / np.linalg.norm(vecarr, axis=-1, keepdims=True)

In [ ]:
from lab.gridspace import gridspace
from lib.gridmesh import gridmesh
from widgets.k3d_view import K3DView

In [ ]:
gshape = lambda gsp: gsp.shape[:-1]
greshape = lambda gsp, dim: (gsp.shape[0], gsp.shape[0], dim)

flat3dify = lambda gsp: np.dstack((gsp, np.zeros(gshape(gsp))))

## Waves

$z = a \cos k x  + a \sin k y$

$f(x, y, z) = z - a \cos k x  + a \sin k y = 0$

> $n = \nabla f = \big[ \frac{\partial f}{\partial x},  \frac{\partial f}{\partial y}, \frac{\partial f}{\partial z} \big]$

$n = [a k \sin k x, - a k \cos k y, 1]$


In [ ]:
def waves_height(gspace: NDArray, amp: float, frq: float = 1.0):
    xx = gspace[..., 0]
    yy = gspace[..., 1]
    a = 0.5 * amp
    k = 2 * math.pi * frq
    return (np.cos(xx * k) + np.sin(yy * k)) * a


def waves_normal(gspace: NDArray, amp: float, frq: float = 1.0):
    xx = gspace[..., 0]
    yy = gspace[..., 1]
    a = 0.5 * amp
    k = 2 * math.pi * frq
    nrm = np.dstack((np.sin(xx * k) * k * a, -np.cos(yy * k) * k * a, np.ones(gshape(gspace))))
    # return nrm
    return normalize(nrm)

---


In [ ]:
# xspace, yspace, gspace = gridspace(-1.0, +1.0, 32, -1.0, +1.0, 32)
# flatvert, indices = gridmesh(gspace)

In [ ]:
# AMP = 0.5
# FRQ = 0.5
# hmap = waves_height(gspace, AMP, FRQ)
# nmap = waves_normal(gspace, AMP, FRQ)

In [ ]:
# hmapvert = np.column_stack((vertices[..., :2], hmap.ravel()))

In [ ]:
# view = K3DView(xspace, yspace, k3plot=dict(renderer="advanced", environment="studio", environment_rotation=math.pi * 0.5))
# # display(view, view.out)
# display(view)

In [ ]:
# view.fields = {"H": hmap, "N": nmap}

In [ ]:
# view.texture("height", attribute=hmap, interpolation=False)
# view.vectors("normals", origins=np.dstack((gspace, hmap)), vectors=nmap * 0.125, use_head=False, line_width=0.001)
# view.mesh("ground", vertices=hmapvert, indices=indices, color=0x808080, flat_shading=False, roughness=0.25)
# view.mesh("plane", vertices=flatvert, indices=indices, normals=nmap, color=0x808080, flat_shading=False, roughness=0.25)

---


## Pimps

Periodic circular bumps around [0.5,0.5] (of each texture grid cell)

$x^2 + y^2 + z^2 - r^2 = 0$

> $n = \nabla f = \big[ \frac{\partial f}{\partial x},  \frac{\partial f}{\partial y}, \frac{\partial f}{\partial z} \big]$

$n = [-2 x, -2 y, -2 z]$


In [ ]:
def pimps_texmap(gspace: NDArray, scale: float, r: float) -> tuple:
    """Make periodic coords in range [-0.5, +0.5] and filter out by proximity to 0"""
    texspace = np.mod(gspace * scale, 1.0) - 0.5
    rsq = r * r
    lattice = dot2(texspace) < rsq
    return texspace, lattice

In [ ]:
def pimps_height(texspace: NDArray, lattice: NDArray, r: float) -> NDArray:
    field = np.zeros(gshape(texspace))
    rsq = r * r

    xy = texspace[lattice]
    z = np.sqrt(rsq - dot2(xy))
    field[lattice] = z
    return field

In [ ]:
def pimps_normal(texspace: NDArray, lattice: NDArray, r: float) -> NDArray:
    field = np.full(greshape(texspace, 3), vec(0.0, 0.0, 1.0))
    rsq = r * r

    xy = texspace[lattice]
    z = np.sqrt(rsq - dot2(xy))
    nrm = np.column_stack((xy, z)) * 2
    field[lattice] = nrm
    return field

---


In [ ]:
# xspace, yspace, gspace = gridspace(-1.0, +1.0, 128, -1.0, +1.0, 128)
# SCALE = 1
# R = 0.25  # inner scale
# texspace, lattice = pimps_texmap(gspace, SCALE, R)
# flatvert, indices = gridmesh(gspace)  # flat mesh

In [ ]:
# hmap = pimps_height(texspace, lattice, R) / SCALE
# nmap = pimps_normal(texspace, lattice, R)

In [ ]:
# hmapvert = np.column_stack((flatvert[..., :2], hmap.ravel()))

In [ ]:
# view = K3DView(xspace, yspace, k3plot=dict(renderer="advanced", environment="studio", environment_rotation=math.pi * 0.5, ao_strength=0))
# # display(view, view.out)
# display(view)

In [ ]:
# view.fields = {"H": hmap, "N": nmap}

In [ ]:
# view.texture("height", attribute=hmap, interpolation=False)
# view.vectors("normals", origins=np.dstack((gspace, hmap)), vectors=nmap * 0.125, use_head=False, line_width=0.001)
# view.mesh("ground", vertices=hmapvert, indices=indices, color=0x808080, flat_shading=False, roughness=0.25)
# view.mesh("plane", vertices=flatvert, indices=indices, normals=nmap, color=0x808080, flat_shading=False, roughness=0.25)

---


# Combining

[RNM = Reoriented Normal Mapping](https://blog.selfshadow.com/publications/blending-in-detail/)

- $n_0 = [0, 0, 1]$ geometric normal
- $n_1 = [x_1, y_2, z_1]$ primary normal map
- $n_2 = [x_2, y_2, z_2]$ secondary normal map

$$n_{combo} \sim n_1' \frac{(n_1'·n_2')}{z_1'} - n_2'$$

- $n_1' = [x_1, y_1, z_1 + 1]$
- $n_2' = [-x_2, -y_2, z_2]$


In [ ]:
def combine_nmap(
    nmap1: NDArray,
    nmap2: NDArray,
):
    x1_ = nmap1[..., 0]
    y1_ = nmap1[..., 1]
    z1_ = nmap1[..., 2] + 1.0
    n1_ = np.dstack((x1_, y1_, z1_))
    x2_ = -nmap2[..., 0]
    y2_ = -nmap2[..., 1]
    z2_ = nmap2[..., 2]
    n2_ = np.dstack((x2_, y2_, z2_))
    cmb = n1_ * (dots(n1_, n2_) / z1_)[..., None] - n2_
    return normalize(cmb)

In [ ]:
isqr3 = 1.0 / math.sqrt(3)
isqr50 = 1.0 / math.sqrt(50)
n0 = nparr(0.0, 0.0, 1.0)
n3 = nparr(isqr3, isqr3, isqr3)
n345 = nparr(3 * isqr50, 4 * isqr50, 5 * isqr50)

assert np.all(np.isclose(combine_nmap(nparr(n0, n0, n3, n345), nparr(n3, n345, n0, n0)), nparr(n3, n345, n3, n345)))

---


In [ ]:
xspace, yspace, gspace = gridspace(-1.0, +1.0, 128, -1.0, +1.0, 128)
flatvert, indices = gridmesh(gspace)

In [ ]:
WAVE_AMP = 0.25
WAVE_FRQ = 0.5
hmap1 = waves_height(gspace, WAVE_AMP, WAVE_FRQ)
nmap1 = waves_normal(gspace, WAVE_AMP, WAVE_FRQ)

PIMP_S = 2
PIMP_R = 0.125
texspace, lattice = pimps_texmap(gspace, PIMP_S, PIMP_R)
hmap2 = pimps_height(texspace, lattice, PIMP_R) / PIMP_S
nmap2 = pimps_normal(texspace, lattice, PIMP_R)

In [ ]:
surf1vert = np.dstack((gspace, hmap1))
surf2vert = np.dstack((gspace, hmap2))

In [ ]:
view = K3DView(
    xspace,
    yspace,
    k3plot=dict(
        camera_fov=15,
        renderer="advanced",
        environment="studio",
        environment_rotation=math.pi * 0.25,
        ao_strength=0,
    ),
)
# display(view, view.out)
display(view)

In [ ]:
view.mesh("ground_1", vertices=surf1vert, indices=indices, color=0x808080, flat_shading=False, roughness=0.5)
view.vectors("normal_1", origins=surf1vert, vectors=nmap1 * 0.125, use_head=False, line_width=0.001)
view.mesh("plane_1", vertices=flatvert, indices=indices, normals=nmap1, color=0x808080, flat_shading=False, roughness=0.5)
view.mesh("ground_2", vertices=surf2vert, indices=indices, color=0x808080, flat_shading=False, roughness=0.5)
view.vectors("normal_2", origins=surf1vert, vectors=nmap2 * 0.125, use_head=False, line_width=0.001)
view.mesh("plane_2", vertices=flatvert, indices=indices, normals=nmap2, color=0x808080, flat_shading=False, roughness=0.5)

In [ ]:
view.fields = {"N1": nmap1, "N2": nmap2}

#### actually modifying mesh


In [ ]:
offset = nmap1 * hmap2[..., None]
combovert = surf1vert + offset.reshape(surf1vert.shape)

In [ ]:
view.mesh("ground_combo", vertices=combovert, indices=indices, color=0x808080, flat_shading=False, roughness=0.5)

#### RNM


In [ ]:
nmap_cmb = combine_nmap(nmap1, nmap2)

In [ ]:
view.fields["Nc"] = nmap_cmb

In [ ]:
view.vectors("normal_combo", origins=combovert, vectors=nmap_cmb * 0.125, use_head=False, line_width=0.001)
view.mesh("plane_combo", vertices=flatvert, indices=indices, normals=nmap_cmb, color=0x808080, flat_shading=False, roughness=0.5)